In [7]:
import yfinance as yf
import pandas as pd
import os
import stock_load as loader 
import stock_features as features
import stock_strategy as strategy
import matplotlib.pyplot as plt
import stock_backtest as backtest

# 1. Loading

In [2]:
START_DATE = "2014-01-01" 
END_DATE = "2025-10-01"
TICKERS_70 = loader.TICKERS_70

In [8]:
# 1. Cấu hình thời gian
# Lấy dư 1 năm (2014) để làm "nguyên liệu" tính toán cho 2015
START_DATE = "2014-01-01"
END_DATE = "2025-10-01"

# 2. Tải dữ liệu
print(f"⏳ Đang tải dữ liệu cho mã...")

df_raw = loader.download_data(
    tickers= loader.TICKERS_70,
    start_date=START_DATE,
    end_date=END_DATE,
)

# Kiểm tra nhanh
if df_raw is not None:
    print("\n--- Dữ liệu thô (5 dòng cuối) ---")
    display(df_raw.tail())

⏳ Đang tải dữ liệu cho mã...
------- Quá trình tải dữ liệu (Fresh Download) ----------------
⬇️ Đang tải dữ liệu từ 2014-01-01 đến 2025-10-01...


[*********************100%***********************]  70 of 70 completed
d:\stock_load.py:63: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df_stacked = raw_data.stack(level=0)


💾 Đã lưu dữ liệu mới vào 'data/stock_data_70.csv'.
-------------------- Hoàn thành -----------------

--- Dữ liệu thô (5 dòng cuối) ---


Open        High         Low       Close   Adj Close  \
Date       Ticker                                                               
2025-09-30 UNH     343.750000  349.320007  342.329987  345.299988  345.299988   
           UNP     237.000000  237.000000  234.820007  236.369995  236.369995   
           V       339.820007  345.609985  338.510010  341.380005  340.705139   
           WMT     103.000000  103.940002  102.720001  103.059998  103.059998   
           XOM     113.349998  113.489998  111.940002  112.750000  111.772377   

                     Volume  
Date       Ticker            
2025-09-30 UNH      7429700  
           UNP      3797500  
           V        8164400  
           WMT     13873400  
           XOM     18076200

# 2. Cleaning & Featuring 

In [9]:
if df_raw is not None:
    # Chạy Pipeline: Fix Missing -> Winsorize -> Calc Indicators
    df_full = features.process_all_tickers(df_raw)

    print("\n--- Dữ liệu sau xử lý (Đã có Features) ---")
    display(df_full.tail())

    # Kiểm tra danh sách các cột đã tạo
    print("\nCác cột hiện có:", df_full.columns.tolist())
else:
    print("Không có dữ liệu để xử lý!")

⚙️ Đang chạy Pipeline: Missing -> Outlier -> Features (GARCH included)...
   -> Chạy GARCH Rolling (2953 phiên)...
   -> Chạy GARCH Rolling (2953 phiên)...
   -> Chạy GARCH Rolling (2953 phiên)...
   -> Chạy GARCH Rolling (2953 phiên)...
   -> Chạy GARCH Rolling (2953 phiên)...
   -> Chạy GARCH Rolling (2953 phiên)...
   -> Chạy GARCH Rolling (2953 phiên)...
   -> Chạy GARCH Rolling (2953 phiên)...
   -> Chạy GARCH Rolling (2953 phiên)...
   -> Chạy GARCH Rolling (2953 phiên)...
   -> Chạy GARCH Rolling (2953 phiên)...
   -> Chạy GARCH Rolling (2953 phiên)...
   -> Chạy GARCH Rolling (2953 phiên)...
   -> Chạy GARCH Rolling (2953 phiên)...
   -> Chạy GARCH Rolling (2953 phiên)...
   -> Chạy GARCH Rolling (2953 phiên)...
   -> Chạy GARCH Rolling (2953 phiên)...
   -> Chạy GARCH Rolling (2953 phiên)...
   -> Chạy GARCH Rolling (2953 phiên)...
   -> Chạy GARCH Rolling (2953 phiên)...
   -> Chạy GARCH Rolling (2953 phiên)...
   -> Chạy GARCH Rolling (2953 phiên)...
   -> Chạy GARCH Rolling

Open        High         Low       Close   Adj Close  \
Date       Ticker                                                               
2025-09-30 UNH     343.750000  349.320007  342.329987  515.429029  427.938306   
           UNP     237.000000  237.000000  234.820007  323.259263  249.393555   
           V       339.820007  345.609985  338.510010  321.465031  295.393826   
           WMT     103.000000  103.940002  102.720001  119.134328   94.439269   
           XOM     113.349998  113.489998  111.940002  184.161661  111.730903   

                     Volume      SMA_50     SMA_200    ROC_12M     ROC_6M  \
Date       Ticker                                                           
2025-09-30 UNH      7429700  469.246667  507.048525  -8.703387  -7.690863   
           UNP      3797500  304.141417  312.435090  -3.826456  -0.176741   
           V        8164400  323.985923  318.900105  27.883136  -0.723478   
           WMT     13873400  115.536417  110.962553  29.851130  13.490588   
           XOM     18076200  180.588456  173.715858   7.480397   0.213338   

                      RSI_14    BB_Lower    BB_Upper           OBV  \
Date       Ticker                                                    
2025-09-30 UNH     60.484818  478.002882  538.007896  4.407058e+08   
           UNP     74.465591  284.321395  325.478487  9.236780e+07   
           V       49.069917  313.824074  329.620134  1.319996e+09   
           WMT     58.805814  115.594201  121.348563  1.607234e+09   
           XOM     50.129971  178.200175  190.975750  9.859910e+07   

                   Vol_SMA_20      ATR_14  GARCH_Vol  
Date       Ticker                                     
2025-09-30 UNH     13192130.0  171.693673       0.01  
           UNP      5093450.0   83.298382       0.01  
           V        6338595.0   22.688423       0.01  
           WMT     15581955.0   16.363927       0.01  
           XOM     16838920.0   72.182588       0.01


Các cột hiện có: ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'SMA_50', 'SMA_200', 'ROC_12M', 'ROC_6M', 'RSI_14', 'BB_Lower', 'BB_Upper', 'OBV', 'Vol_SMA_20', 'ATR_14', 'GARCH_Vol']


# 3. Exporting 

In [ ]:
df_full.to_csv('data/stock_features_70.csv')

In [13]:
#Double check loading saved file
df_feature = loader.load_local_data("data/stock_features_70.csv")
df_feature.head()

📂 Đang đọc dữ liệu từ đĩa: data/stock_features_70.csv...


Open       High        Low      Close  Adj Close  \
Date       Ticker                                                          
2015-01-02 AAPL    27.847500  27.860001  26.837500  27.990911  24.309819   
           ABBV    65.440002  66.400002  65.440002  69.132147  42.780787   
           ADBE    72.699997  73.199997  71.889999  67.814061  67.814061   
           AEP     60.880001  61.240002  60.389999  63.568169  41.371323   
           AMD      2.670000   2.670000   2.670000   3.016693   3.016693   

                      Volume     SMA_50    SMA_200    ROC_12M     ROC_6M  \
Date       Ticker                                                          
2015-01-02 AAPL    212818400  28.556430  24.665783  41.692826  17.356357   
           ABBV      5086100  68.501915  59.242635  32.997591  14.488042   
           ADBE      2349200  68.899243  66.867141  14.376893  -5.148512   
           AEP       2007400  60.504253  55.540053  37.802227  15.904348   
           AMD             0   3.026183   3.905983 -23.628034 -30.685501   

                      RSI_14   BB_Lower   BB_Upper           OBV   Vol_SMA_20  \
Date       Ticker                                                               
2015-01-02 AAPL    42.667288  27.467814  29.797983  3.830646e+09  188114580.0   
           ABBV    48.317797  67.033924  73.566935  2.136187e+08    7272525.0   
           ADBE    42.135592  67.180823  71.523134  5.182550e+07    3764950.0   
           AEP     60.488939  58.927132  65.222558  1.003562e+08    2427780.0   
           AMD     52.336359   2.764299   3.100739 -1.081590e+07   12857955.0   

                     ATR_14  GARCH_Vol  
Date       Ticker                       
2015-01-02 AAPL    0.982929       0.01  
           ABBV    3.888274       0.01  
           ADBE    4.399609       0.01  
           AEP     2.733794       0.01  
           AMD     0.380670       0.01